# 2.5 — Huấn luyện DL cuối và tạo submission

Notebook đọc mô hình được chọn ở task 2.4, fit preprocessing và huấn luyện lại trên toàn bộ train.csv, rồi dự đoán xác suất is_fraud cho test.csv. Không dùng test labels và không tự nộp lên leaderboard.

Đầu ra submission.csv phải khớp sample_submission.csv về schema, số dòng và ID; is_fraud là xác suất trong [0,1].

## 2.5.1 — Nạp quyết định validation và dữ liệu Kaggle

Đính kèm output task 2.4 để đọc selected_model.json cùng competition data. Số epoch được chốt từ validation, không chọn bằng test.

In [ ]:
import importlib.util
import json
import random
import subprocess
import sys
import time
from pathlib import Path

if importlib.util.find_spec("pytorch_tabnet") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pytorch-tabnet==4.1.0"])

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from pytorch_tabnet.tab_model import TabNetClassifier

SEED = 9
ID_COL, TARGET = "id", "is_fraud"
DATA_DIR = Path("/kaggle/input/competitions/playground-series-s6e11")
OUTPUT_DIR = Path("/kaggle/working/task_2_5_final")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

selection_files = list(Path("/kaggle/input").rglob("selected_model.json"))
if len(selection_files) != 1:
    raise FileNotFoundError("Attach task 2.4 output to find selected_model.json.")
decision = json.loads(selection_files[0].read_text(encoding="utf-8"))
selected_name = decision["selected_model"]
epochs = int(decision["best_epoch"])
supported = {"MLP baseline", "MLP weighted", "MLP embedding", "TabNet baseline"}
if selected_name not in supported or epochs < 1:
    raise ValueError(f"Invalid selected model or epoch: {selected_name}, {epochs}.")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
if TARGET not in train or ID_COL not in train or ID_COL not in test:
    raise ValueError("Required columns are missing from train/test.")
if train[ID_COL].duplicated().any() or test[ID_COL].duplicated().any():
    raise ValueError("Duplicate IDs detected in train or test.")
if list(sample_submission.columns) != [ID_COL, TARGET]:
    raise ValueError(f"Invalid sample_submission schema: {sample_submission.columns.tolist()}")

print("Selected model:", selected_name)
print("Validation metrics:", decision["validation_metrics"])
print("Refit epochs:", epochs)
print("Train/test rows:", len(train), len(test), "| device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2.5.2 — Fit preprocessing trên toàn bộ train

Chọn preprocessing theo đúng họ mô hình đã được chọn. Scaler, category mappings và encoder chỉ fit trên train; sau đó áp dụng cùng phép biến đổi lên test. Cột id và target không là features.

In [ ]:
y_train = train[TARGET].astype(np.int64).to_numpy()
positive_count = int(y_train.sum())
negative_count = int(len(y_train) - positive_count)
pos_weight = negative_count / positive_count

X_train_raw = train.drop(columns=[ID_COL, TARGET]).copy()
X_test_raw = test.drop(columns=[ID_COL]).copy()
bool_cols = X_train_raw.select_dtypes(include=["bool"]).columns.tolist()
for frame in (X_train_raw, X_test_raw):
    frame[bool_cols] = frame[bool_cols].astype("int8")
cat_cols = X_train_raw.select_dtypes(include=["object", "category", "string"]).columns.tolist()
num_cols = [col for col in X_train_raw.columns if col not in cat_cols]
for frame in (X_train_raw, X_test_raw):
    for col in cat_cols:
        frame[col] = frame[col].astype("string").fillna("__MISSING__")
    for col in num_cols:
        frame[col] = pd.to_numeric(frame[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
numeric_medians = X_train_raw[num_cols].median()
X_train_raw[num_cols] = X_train_raw[num_cols].fillna(numeric_medians)
X_test_raw[num_cols] = X_test_raw[num_cols].fillna(numeric_medians)

category_mappings = {}
if selected_name in {"MLP baseline", "MLP weighted"}:
    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", StandardScaler(), num_cols),
            ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32), cat_cols),
        ],
        remainder="drop",
    )
    X_train = preprocessor.fit_transform(X_train_raw).astype(np.float32)
    X_test = preprocessor.transform(X_test_raw).astype(np.float32)
    joblib.dump({"preprocessor": preprocessor, "numeric_medians": numeric_medians.to_dict(), "numeric_columns": num_cols, "categorical_columns": cat_cols, "seed": SEED}, OUTPUT_DIR / "preprocessor.joblib")
else:
    scaler = StandardScaler()
    X_train_num = scaler.fit_transform(X_train_raw[num_cols]).astype(np.float32)
    X_test_num = scaler.transform(X_test_raw[num_cols]).astype(np.float32)
    X_train_cat = np.zeros((len(train), len(cat_cols)), dtype=np.int64)
    X_test_cat = np.zeros((len(test), len(cat_cols)), dtype=np.int64)
    cat_dims, cat_emb_dims = [], []
    for i, col in enumerate(cat_cols):
        categories = sorted(X_train_raw[col].astype(str).unique().tolist())
        mapping = {value: index + 1 for index, value in enumerate(categories)}
        category_mappings[col] = mapping
        X_train_cat[:, i] = X_train_raw[col].astype(str).map(mapping).to_numpy(dtype=np.int64)
        X_test_cat[:, i] = X_test_raw[col].astype(str).map(mapping).fillna(0).to_numpy(dtype=np.int64)
        cat_dims.append(len(mapping) + 1)
        cat_emb_dims.append(min(16, max(2, int(np.ceil(np.sqrt(len(mapping) + 1))))))
    joblib.dump(
        {"scaler": scaler, "numeric_medians": numeric_medians.to_dict(),
         "category_mappings": category_mappings, "numeric_columns": num_cols,
         "categorical_columns": cat_cols, "categorical_dimensions": cat_dims,
         "embedding_dimensions": cat_emb_dims, "seed": SEED},
        OUTPUT_DIR / "preprocessor.joblib",
    )
    if selected_name == "TabNet baseline":
        X_train = np.concatenate([X_train_num, X_train_cat.astype(np.float32)], axis=1)
        X_test = np.concatenate([X_test_num, X_test_cat.astype(np.float32)], axis=1)
        cat_idxs = list(range(len(num_cols), len(num_cols) + len(cat_cols)))

print("Feature counts:", len(num_cols), "numeric/boolean;", len(cat_cols), "categorical")
print("Fraud rate in full train:", round(float(y_train.mean()), 6))
print("Full-train pos_weight:", round(pos_weight, 4))

## 2.5.2 — Refit ứng viên được chọn

Giữ kiến trúc và optimizer của thí nghiệm đã chọn. MLP refit đúng số epoch chốt trên validation. TabNet dùng max_epochs bằng số epoch đã chọn và tắt early stopping vì final refit không có validation set.

In [ ]:
class EmbeddingMLP(nn.Module):
    def __init__(self, n_numeric, dimensions, embedding_dimensions):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(cardinality, embedding_dim)
            for cardinality, embedding_dim in zip(dimensions, embedding_dimensions)
        ])
        self.network = nn.Sequential(
            nn.Linear(n_numeric + sum(embedding_dimensions), 128), nn.ReLU(), nn.Dropout(0.20),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.10), nn.Linear(64, 1),
        )

    def forward(self, numeric, categorical):
        embedded = [layer(categorical[:, i]) for i, layer in enumerate(self.embeddings)]
        return self.network(torch.cat([numeric, *embedded], dim=1))


def fit_mlp(model, arrays, weighted=False):
    model = model.to(DEVICE)
    tensors = []
    for array in arrays:
        array = np.asarray(array)
        tensor_dtype = torch.long if np.issubdtype(array.dtype, np.integer) else torch.float32
        tensors.append(torch.as_tensor(array, dtype=tensor_dtype))
    y_tensor = torch.as_tensor(y_train.reshape(-1, 1), dtype=torch.float32)
    loader = DataLoader(
        TensorDataset(*tensors, y_tensor),
        batch_size=4096, shuffle=True, num_workers=0,
        pin_memory=torch.cuda.is_available(),
        generator=torch.Generator().manual_seed(SEED),
    )
    criterion = nn.BCEWithLogitsLoss()
    if weighted:
        criterion = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([pos_weight], dtype=torch.float32, device=DEVICE)
        )
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, seen = 0.0, 0
        for batch in loader:
            *features, target = batch
            features = [x.to(DEVICE, non_blocking=True) for x in features]
            target = target.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(*features), target)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += float(loss.item()) * len(target)
            seen += len(target)
        history.append({"epoch": epoch, "train_loss": total_loss / seen})
        print(f"Epoch {epoch:02d}/{epochs} | train loss={total_loss / seen:.5f}")
    model.eval()
    torch.save(
        {"state_dict": model.state_dict(), "model_name": selected_name,
         "seed": SEED, "epochs": epochs, "pos_weight": pos_weight if weighted else None},
        OUTPUT_DIR / "best_model.pt",
    )
    pd.DataFrame(history).to_csv(OUTPUT_DIR / "training_history.csv", index=False)
    return model


training_start = time.time()
if selected_name in {"MLP baseline", "MLP weighted"}:
    final_model = fit_mlp(
        nn.Sequential(
            nn.Linear(X_train.shape[1], 128), nn.ReLU(), nn.Dropout(0.20),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.10), nn.Linear(64, 1),
        ),
        [X_train],
        weighted=(selected_name == "MLP weighted"),
    )
elif selected_name == "MLP embedding":
    final_model = fit_mlp(
        EmbeddingMLP(len(num_cols), cat_dims, cat_emb_dims),
        [X_train_num, X_train_cat],
    )
else:
    final_model = TabNetClassifier(
        n_d=16, n_a=16, n_steps=4, gamma=1.3, lambda_sparse=1e-4,
        cat_idxs=cat_idxs, cat_dims=cat_dims, cat_emb_dim=1,
        optimizer_fn=torch.optim.Adam, optimizer_params={"lr": 2e-2},
        mask_type="entmax", seed=SEED, verbose=10,
    )
    final_model.fit(
        X_train=X_train, y_train=y_train, max_epochs=epochs, patience=0,
        batch_size=4096, virtual_batch_size=512, num_workers=0, drop_last=False,
    )
    final_model.save_model(str(OUTPUT_DIR / "tabnet_final"))
training_seconds = time.time() - training_start
print("Final full-train training seconds:", round(training_seconds, 1))

## 2.5.3–2.5.6 — Dự đoán và kiểm tra submission

Tạo xác suất fraud cho toàn bộ test. Ghép theo ID vào sample_submission để giữ đúng thứ tự; kiểm tra schema, số dòng, ID, giá trị thiếu, tính hữu hạn và khoảng [0,1].

In [ ]:
if selected_name == "TabNet baseline":
    test_probability = final_model.predict_proba(X_test)[:, 1].astype(np.float64)
else:
    final_model.eval()
    probability_parts = []
    with torch.inference_mode():
        for start in range(0, len(test), 8192):
            stop = min(start + 8192, len(test))
            if selected_name in {"MLP baseline", "MLP weighted"}:
                batch_features = [torch.as_tensor(X_test[start:stop], dtype=torch.float32, device=DEVICE)]
            else:
                batch_features = [
                    torch.as_tensor(X_test_num[start:stop], dtype=torch.float32, device=DEVICE),
                    torch.as_tensor(X_test_cat[start:stop], dtype=torch.long, device=DEVICE),
                ]
            logits = final_model(*batch_features).squeeze(1)
            probability_parts.extend(torch.sigmoid(logits).cpu().numpy().tolist())
    test_probability = np.asarray(probability_parts, dtype=np.float64)

if len(test_probability) != len(test):
    raise ValueError("Prediction count does not match test row count.")
if not np.isfinite(test_probability).all() or ((test_probability < 0) | (test_probability > 1)).any():
    raise ValueError("Submission probabilities are missing, non-finite, or outside [0, 1].")
if sample_submission[ID_COL].duplicated().any() or set(sample_submission[ID_COL]) != set(test[ID_COL]):
    raise ValueError("sample_submission IDs do not match test.csv.")

probability_by_id = pd.Series(test_probability, index=test[ID_COL].to_numpy())
submission = sample_submission[[ID_COL]].copy()
submission[TARGET] = submission[ID_COL].map(probability_by_id)
submission = submission[[ID_COL, TARGET]]
if submission.isna().any().any() or len(submission) != len(sample_submission):
    raise ValueError("Submission IDs/predictions are missing or row count is wrong.")
if list(submission.columns) != [ID_COL, TARGET]:
    raise ValueError(f"Invalid submission schema: {submission.columns.tolist()}")

submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)
print("Submission:", submission_path)
print("Rows/columns:", submission.shape, submission.columns.tolist())
print("Probability min/max/mean:",
      float(submission[TARGET].min()), float(submission[TARGET].max()), float(submission[TARGET].mean()))
display(submission.head())

## 2.5.7–2.5.8 — Lưu metadata và báo cáo tái lập

Lưu cấu hình cuối, metric validation, seed, số epoch, môi trường, số dòng test và các kiểm tra định dạng. Báo cáo không thêm leaderboard score vì file chưa được nộp.

In [ ]:
metadata = {
    "task": "2.5 final DL submission",
    "selected_model": selected_name,
    "selection_rule": decision["selection_rule"],
    "validation_metrics": decision["validation_metrics"],
    "seed": SEED,
    "epochs": epochs,
    "full_train_rows": int(len(train)),
    "positive_count_full_train": positive_count,
    "positive_rate_full_train": float(y_train.mean()),
    "full_train_pos_weight": float(pos_weight),
    "numeric_columns": num_cols,
    "categorical_columns": cat_cols,
    "training_seconds": float(training_seconds),
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "test_rows": int(len(test)),
    "submission_path": str(submission_path),
    "submission_rows": int(len(submission)),
    "submission_columns": submission.columns.tolist(),
    "probability_min": float(submission[TARGET].min()),
    "probability_max": float(submission[TARGET].max()),
    "test_labels_used": False,
    "competition_submission_uploaded": False,
}
with open(OUTPUT_DIR / "model_metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

report_lines = [
    "# Nhánh DL — báo cáo cuối",
    "",
    f"- Mô hình cuối: {selected_name}",
    f"- Validation seed 9: ROC-AUC {decision['validation_metrics']['roc_auc']:.6f}; Average Precision {decision['validation_metrics']['average_precision']:.6f}.",
    f"- Huấn luyện lại trên {len(train):,} dòng train trong {epochs} epoch.",
    f"- Test: {len(test):,} dòng; submission: {len(submission):,} dòng, cột id/is_fraud.",
    f"- Xác suất submission nằm trong [{metadata['probability_min']:.6f}, {metadata['probability_max']:.6f}].",
    "- Không dùng test labels; không dùng leaderboard để chọn mô hình.",
    "- Giới hạn: so sánh dựa trên một holdout validation chung; không chạy cross-validation.",
]
(OUTPUT_DIR / "task_2_5_report.md").write_text("\n".join(report_lines), encoding="utf-8")
print("Metadata and report saved in:", OUTPUT_DIR)